In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import GridSearchCV

import warnings
warnings.filterwarnings("ignore")

In [10]:
DATA_PATH = os.path.join("..","datasets","prepared_data","telco_data.csv")
telco = pd.read_csv(DATA_PATH)

In [12]:
telco.isnull().sum()

tenure              0
contract            0
internetservice     0
paymentmethod       0
onlinesecurity      0
techsupport         0
phoneservice        0
paperlessbilling    0
churn               0
dtype: int64

In [13]:
numerical_features = telco.select_dtypes(exclude='object')
categorical_features = telco.select_dtypes(include='object')

In [14]:
# Train, Validation and Test set
df_train_full, df_test = train_test_split(telco, test_size=0.2, random_state=12)
df_train, df_val = train_test_split(df_train_full, test_size=0.33, random_state=3)

y_train = df_train.churn.values
y_val = df_val.churn.values

del df_train['churn']
del df_val['churn']

In [15]:
# # Transorming train set to dict
train_dict = df_train[categorical_features + numerical_features].to_dict(orient='records')

# Vectorizer
dv = DictVectorizer(sparse=False)   

# Transforming dict train set to vector
X_train = dv.fit_transform(train_dict)  

In [16]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

models = {'svc': SVC(), 
          'logistic': LogisticRegression(),
          'random_forest': RandomForestClassifier(),
          'knn': KNeighborsClassifier()}

def evaluate_models(models:dict, X:pd.DataFrame, y:pd.Series, cv:int) -> pd.DataFrame:
    n_models = len(models)
    scores=[]
    for name, model in models.items():
        y_train_pred = cross_val_predict(model, X, y, cv=cv)
        accuracy = np.mean(accuracy_score(y, y_train_pred)).round(2)
        precision = np.mean(precision_score(y, y_train_pred)).round(2)
        recall = np.mean(recall_score(y, y_train_pred)).round(2)
        f1 = np.mean(f1_score(y, y_train_pred)).round(2)

        model_scores = np.array([name, accuracy, precision, recall, f1])
        scores.append(np.array(model_scores))
          
    scores_df = pd.DataFrame(scores, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1']); scores_df.set_index('Model', inplace=True)
    scores_df.sort_values(by='F1', ascending=False)
    return scores_df

In [17]:
evaluate_models(models, X_train, y_train, 5)

ValueError: Input X contains NaN.
SVC does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [18]:
X_train

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])

In [ ]:
model = SVC()

In [ ]:
%run ../auxiliar_functions/rocResults.ipynb

rocResults(model, X_train, y_train, 5)

In [ ]:
params = {
    'penalty':['l1', 'l2', 'elasticnet', 'none'],
    'C':[0.001, 0.01, 0.1, 0.5, 1, 10]}

grid_search = GridSearchCV(model, param_grid=params, cv=3)

grid_search.fit(X_train, y_train)
grid_search.best_params_

In [ ]:
rocResults(grid_search.best_estimator_, X_train, y_train, 5)